# Statistical Validity of Backbone Comparison

**Primary test: Cluster bootstrap** — resample cameras (deployment locations) with replacement.
This accounts for spatial dependence (images from the same camera are correlated) and gives
properly wider CIs than image-level bootstrap. Also directly shows robustness to which
camera locations you happen to evaluate on — a stronger statement for a camera-trap setting.

**Complementary tests:**
1. **Image-level bootstrap** (secondary) — resample individual images; narrower CIs, shown for comparison.
2. **Per-camera Wilcoxon signed-rank** — non-parametric paired test on per-camera F1. Answers: *across deployment locations, does one backbone consistently outperform?*
3. **McNemar's test** — paired binary test (correct/incorrect per image). Answers: *at the image level, is one model significantly better?*

All pairwise comparisons (3 pairs) with **Holm-Bonferroni** correction for multiple testing.

_Bootstrap-count / camera-threshold convergence checks and the methodological-justification figure now live in `bootstrap_methodology.ipynb`._


In [ ]:
import json
import sys
from collections import Counter, defaultdict
from itertools import combinations
from pathlib import Path

import yaml

sys.path.insert(0, "../../src")

import matplotlib.pyplot as plt
import numpy as np
import torch
from scipy import stats
from sklearn.metrics import f1_score
from statsmodels.stats.contingency_tables import mcnemar
from tqdm import tqdm

from data.iwildcam import iWildCamDataModule
from models.base_classifier import BaseClassifier

OUTPUT_DIR = Path("./evaluation_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def _best_checkpoint(base: Path, model_dir: str) -> str:
    """Resolve the winning LR's checkpoint via results.yaml instead of hardcoding it."""
    sweep_dir = base / f"sweep_{model_dir}"
    with open(sweep_dir / "results.yaml") as f:
        results = yaml.safe_load(f)
    lr_dir = sweep_dir / f"lr_{results['best_lr']:.0e}"
    ckpts = [p for p in lr_dir.glob("*.ckpt") if p.name != "last.ckpt"]
    assert len(ckpts) == 1, (
        f"expected exactly one checkpoint in {lr_dir}, found {ckpts}"
    )
    return str(ckpts[0])


BASE = Path("../../checkpoints/cycle1_sweep")

MODELS = {
    "DINOv2 ViT-B/14": _best_checkpoint(BASE, "dinov2_vitb14"),
    "ResNet50 pretrained": _best_checkpoint(BASE, "resnet50_pretrained"),
    "timm ViT-B/16": _best_checkpoint(BASE, "timm_vit_base_patch16_224"),
}

MODEL_PAIRS = list(combinations(MODELS.keys(), 2))
N_PAIRS = len(MODEL_PAIRS)  # 3

# Legacy aliases (used by some analysis cells)
MODEL_A = "DINOv2 ViT-B/14"
MODEL_B = "ResNet50 pretrained"

RARE_THRESHOLD = 10
COMMON_THRESHOLD = 100
NON_SPECIES = {0}
BLANK_CLASS = 0
MIN_CAM_IMAGES = 15

# Efron, B. & Tibshirani, R.J. (1993), An Introduction to the Bootstrap
N_BOOTSTRAP = 2000
SEED = 42
rng = np.random.default_rng(SEED)


def holm_bonferroni(pvalues, alpha=0.05):
    """
    Holm-Bonferroni correction for multiple comparisons.
    Returns list of (adjusted_p, significant) for each input p-value.
    """
    n = len(pvalues)
    indexed = sorted(enumerate(pvalues), key=lambda x: x[1])
    adjusted = [None] * n
    cummax = 0.0
    for rank, (orig_idx, p) in enumerate(indexed):
        adj_p = min(p * (n - rank), 1.0)
        cummax = max(cummax, adj_p)
        adjusted[orig_idx] = (cummax, cummax <= alpha)
    return adjusted

In [ ]:
dm = iWildCamDataModule(
    root_dir=Path("/path/to/iwildcam_224/"),
    metadata_path=Path("metadata/metadata/iwildcam2022_train_annotations.json"),
    image_dir="train/train",
    batch_size=64,
    num_workers=4,
)
dm.setup("test")

full_ds = dm.test_dataset.dataset
test_idx_list = list(dm.test_dataset.indices)

sample_locations = []
for idx in test_idx_list:
    img_id = full_ds.valid_image_ids[idx]
    sample_locations.append(full_ds.image_lookup[img_id]["location"])
sample_locations = np.array(sample_locations)

train_class_dist = dm.train_class_dist
test_class_dist = dm.test_class_dist
eligible_classes = set(train_class_dist) & set(test_class_dist)

rare_classes = {
    c for c in eligible_classes if train_class_dist.get(c, 0) < RARE_THRESHOLD
}
common_classes = {
    c
    for c in eligible_classes
    if RARE_THRESHOLD <= train_class_dist.get(c, 0) <= COMMON_THRESHOLD
}
frequent_classes = {
    c for c in eligible_classes if train_class_dist.get(c, 0) > COMMON_THRESHOLD
}

# Gupta et al. (2019)
lvis_groups = {
    f"Rare (<{RARE_THRESHOLD})": rare_classes,
    f"Common ({RARE_THRESHOLD}–{COMMON_THRESHOLD})": common_classes,
    f"Frequent (>{COMMON_THRESHOLD})": frequent_classes,
}

lvis_segments = {
    "Head (>100)": frequent_classes,
    "Medium (10–100)": common_classes,
    "Tail (<10)": rare_classes,
}

# Gadot et al. (2024) To crop or not to crop
species_classes = eligible_classes - NON_SPECIES

cam_counts = Counter(sample_locations)
qualifying_cams = {cam for cam, n in cam_counts.items() if n >= MIN_CAM_IMAGES}
print(f"Total val cameras: {len(cam_counts)}")
print(f"Qualifying cameras (>={MIN_CAM_IMAGES} images): {len(qualifying_cams)}")
print(
    f"Images in qualifying cameras: {sum(cam_counts[c] for c in qualifying_cams)} "
    f"({sum(cam_counts[c] for c in qualifying_cams) / len(sample_locations):.1%} of val)"
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
test_loader = dm.test_dataloader()

all_results = {}
for model_name, ckpt_path in MODELS.items():
    print(f"\nEvaluating {model_name}...")
    model = BaseClassifier.load_from_checkpoint(ckpt_path)
    model.eval().to(device)

    preds_list, targets_list = [], []
    with torch.no_grad():
        for images, targets in tqdm(test_loader, desc=model_name):
            logits = model(images.to(device))
            preds_list.extend(logits.argmax(1).cpu().tolist())
            targets_list.extend(targets.tolist())

    preds = np.array(preds_list)
    targets = np.array(targets_list)

    mask = np.isin(targets, list(eligible_classes))
    preds_e = preds[mask]
    targets_e = targets[mask]
    locs_e = sample_locations[mask]

    all_results[model_name] = dict(
        preds=preds_e,
        targets=targets_e,
        locations=locs_e,
    )

    f1m = f1_score(targets_e, preds_e, average="macro", zero_division=0)
    print(f"  F1 macro (eligible classes): {f1m:.4f}")
    del model
    torch.cuda.empty_cache()

In [ ]:
# PRIMARY TEST: CLUSTER BOOTSTRAP (resample cameras)
#
# Rationale: images from the same camera are spatially correlated.
# Image-level bootstrap treats every image as iid and underestimates
# uncertainty. Cluster bootstrap resamples deployment locations with
# replacement, giving properly wider CIs that reflect how performance
# would vary across different sets of camera locations.
#
# All pairwise comparisons with Holm-Bonferroni correction.

qual_mask = np.isin(all_results[MODEL_A]["locations"], list(qualifying_cams))

filtered_results = {}
for model_name, res in all_results.items():
    filtered_results[model_name] = dict(
        targets=res["targets"][qual_mask],
        preds=res["preds"][qual_mask],
        locations=res["locations"][qual_mask],
    )

n_qual_cams = len(np.unique(filtered_results[MODEL_A]["locations"]))
n_qual_imgs = qual_mask.sum()
print(f"Filtered to {n_qual_cams} qualifying cameras ({n_qual_imgs} images)\n")


def cluster_bootstrap_f1(targets, preds, locations, n_boot=N_BOOTSTRAP, groups=None):
    """
    Bootstrap by resampling cameras (clusters) with replacement.
    Returns: (point_estimate, ci_95, boot_samples, group_results)
    """
    cameras = np.unique(locations)
    cam_idx = {cam: np.where(locations == cam)[0] for cam in cameras}

    boot_f1 = []
    boot_groups = defaultdict(list)

    for _ in range(n_boot):
        sampled_cams = rng.choice(cameras, size=len(cameras), replace=True)
        idx = np.concatenate([cam_idx[c] for c in sampled_cams])
        t_b, p_b = targets[idx], preds[idx]
        boot_f1.append(f1_score(t_b, p_b, average="macro", zero_division=0))
        if groups:
            for gname, gcls in groups.items():
                gm = np.isin(t_b, list(gcls))
                boot_groups[gname].append(
                    f1_score(t_b[gm], p_b[gm], average="macro", zero_division=0)
                    if gm.sum() > 0
                    else 0.0
                )

    boot_f1 = np.array(boot_f1)
    point = f1_score(targets, preds, average="macro", zero_division=0)
    ci = np.percentile(boot_f1, [2.5, 97.5])

    group_out = {}
    if groups:
        for gname, vals in boot_groups.items():
            vals = np.array(vals)
            gmask = np.isin(targets, list(groups[gname]))
            gpt = (
                f1_score(targets[gmask], preds[gmask], average="macro", zero_division=0)
                if gmask.sum() > 0
                else 0.0
            )
            group_out[gname] = (gpt, np.percentile(vals, [2.5, 97.5]), vals)

    return point, ci, boot_f1, group_out


print(f"Cluster bootstrap: {N_BOOTSTRAP} resamples of {n_qual_cams} cameras\n")
clust_results = {}

for model_name, res in filtered_results.items():
    point, ci, samples, groups_out = cluster_bootstrap_f1(
        res["targets"], res["preds"], res["locations"], groups=lvis_groups
    )
    clust_results[model_name] = dict(
        point=point, ci=ci, samples=samples, groups=groups_out
    )
    print(f"{model_name}")
    print(
        f"  F1 macro: {point:.4f}  95% CI [{ci[0]:.4f}, {ci[1]:.4f}]  width={ci[1] - ci[0]:.4f}"
    )
    for gname, (gp, gci, _) in groups_out.items():
        print(f"  {gname}: {gp:.4f}  95% CI [{gci[0]:.4f}, {gci[1]:.4f}]")
    print()

cameras = np.unique(filtered_results[MODEL_A]["locations"])
cam_idx_a = {
    cam: np.where(filtered_results[MODEL_A]["locations"] == cam)[0] for cam in cameras
}

pair_gap_results = {}

for pair_i, (m_a, m_b) in enumerate(MODEL_PAIRS):
    res_a = filtered_results[m_a]
    res_b = filtered_results[m_b]

    paired_rng = np.random.default_rng(SEED + 100 + pair_i)
    gap_overall, gap_groups = [], defaultdict(list)

    for _ in range(N_BOOTSTRAP):
        sampled_cams = paired_rng.choice(cameras, size=len(cameras), replace=True)
        idx = np.concatenate([cam_idx_a[c] for c in sampled_cams])
        ta, pa = res_a["targets"][idx], res_a["preds"][idx]
        tb, pb = res_b["targets"][idx], res_b["preds"][idx]
        f1a = f1_score(ta, pa, average="macro", zero_division=0)
        f1b = f1_score(tb, pb, average="macro", zero_division=0)
        gap_overall.append(f1a - f1b)
        for gname, gcls in lvis_groups.items():
            gm = np.isin(ta, list(gcls))
            gap_groups[gname].append(
                f1_score(ta[gm], pa[gm], average="macro", zero_division=0)
                - f1_score(tb[gm], pb[gm], average="macro", zero_division=0)
                if gm.sum() > 0
                else 0.0
            )

    gap_overall = np.array(gap_overall)
    ci_gap = np.percentile(gap_overall, [2.5, 97.5])
    point_gap = clust_results[m_a]["point"] - clust_results[m_b]["point"]
    p_raw = 2 * min((gap_overall <= 0).mean(), (gap_overall >= 0).mean())

    pair_gap_results[(m_a, m_b)] = dict(
        gap=gap_overall,
        ci=ci_gap,
        point=point_gap,
        p_raw=p_raw,
        gap_groups={g: np.array(v) for g, v in gap_groups.items()},
    )

# Holm-Bonferroni correction on overall p-values
raw_ps = [pair_gap_results[pair]["p_raw"] for pair in MODEL_PAIRS]
adjusted = holm_bonferroni(raw_ps)

print("=" * 80)
print(f"PAIRED CLUSTER BOOTSTRAP — ALL PAIRS (Holm-Bonferroni, {N_PAIRS} comparisons)")
print("=" * 80)

for idx, (m_a, m_b) in enumerate(MODEL_PAIRS):
    r = pair_gap_results[(m_a, m_b)]
    adj_p, sig = adjusted[idx]
    mark = "✓" if sig else "✗"
    print(f"\n{m_a}  vs  {m_b}")
    print(f"  F1 gap: {r['point']:+.4f}  95% CI [{r['ci'][0]:+.4f}, {r['ci'][1]:+.4f}]")
    print(f"  p-value (raw): {r['p_raw']:.4f}   p-value (Holm): {adj_p:.4f}")
    print(
        f"  {mark} {'Significant' if sig else 'NOT significant'} at α=0.05 (Holm-corrected)"
    )

    # Per-group gaps
    for gname in lvis_groups:
        g_arr = r["gap_groups"][gname]
        g_ci = np.percentile(g_arr, [2.5, 97.5])
        gp = (
            clust_results[m_a]["groups"][gname][0]
            - clust_results[m_b]["groups"][gname][0]
        )
        g_p = 2 * min((g_arr <= 0).mean(), (g_arr >= 0).mean())
        s = "✓" if g_ci[0] > 0 else "✗"
        print(
            f"  {s} {gname}: gap={gp:+.4f}  CI [{g_ci[0]:+.4f}, {g_ci[1]:+.4f}]  p={g_p:.4f}"
        )

In [ ]:
# PRIMARY TEST: CLUSTER BOOTSTRAP (resample cameras)
#
# Rationale: images from the same camera are spatially correlated.
# Image-level bootstrap treats every image as iid and underestimates
# uncertainty. Cluster bootstrap resamples deployment locations with
# replacement, giving properly wider CIs that reflect how performance
# would vary across different sets of camera locations.
#
# All pairwise comparisons with Holm-Bonferroni correction.
#
# METRIC DEFINITIONS (must match the descriptive table):
#   Overall macro-F1 — all images, sklearn's inferred label set.
#   Segment F1       — images subset to the segment, averaged over the
#                      segment's own classes only. Out-of-group predicted
#                      classes are excluded from the denominator, so segment
#                      scores do NOT reconcile to the overall macro-F1 and
#                      exclude cross-segment false positives.

qual_mask = np.isin(all_results[MODEL_A]["locations"], list(qualifying_cams))
rng = np.random.default_rng(SEED)

filtered_results = {}
for model_name, res in all_results.items():
    filtered_results[model_name] = dict(
        targets=res["targets"][qual_mask],
        preds=res["preds"][qual_mask],
        locations=res["locations"][qual_mask],
    )

n_qual_cams = len(np.unique(filtered_results[MODEL_A]["locations"]))
n_qual_imgs = qual_mask.sum()
print(f"Filtered to {n_qual_cams} qualifying cameras ({n_qual_imgs} images)\n")


# Label set is left to sklearn's inference (union of y_true and y_pred), which
# is the definition behind the published numbers. Do NOT pin `labels=` to the
# full class list: classes absent from a resample would then contribute an
# F1 of exactly 0 to the macro average, biasing every resample downward while
# leaving the point estimate untouched.
def macro_f1(t, p):
    return f1_score(t, p, average="macro", zero_division=0)


def group_f1(t, p, gcls):
    """
    Segment F1 matching the descriptive table: images already subset to the
    segment, averaged over the segment's own classes only.

    Labels are pinned to group classes PRESENT in `t`, not the full group list.
    A class absent from a resample is undefined, not zero — pinning the full
    list would add fabricated zeros to every resample and push the CI below
    the point estimate. On the full data every group class is present, so this
    reduces to the descriptive-table definition exactly.
    """
    labels = sorted(set(gcls) & set(np.unique(t)))
    if not labels:
        return np.nan
    return f1_score(t, p, labels=labels, average="macro", zero_division=0)


def boot_p(arr):
    """
    Two-sided bootstrap p-value by CI inversion, ignoring NaN (dropped) draws.
    Add-one estimator: avoids reporting p=0 (resolution floor is 1/n_boot), and
    min(1.0, ...) guards against double-counting exact zeros, which would
    otherwise let p exceed 1.
    """
    a = arr[~np.isnan(arr)]
    if len(a) == 0:
        return np.nan
    p_lo = (np.sum(a <= 0) + 1) / (len(a) + 1)
    p_hi = (np.sum(a >= 0) + 1) / (len(a) + 1)
    return min(1.0, 2 * min(p_lo, p_hi))


def cluster_bootstrap_f1(targets, preds, locations, n_boot=N_BOOTSTRAP, groups=None):
    """
    Bootstrap by resampling cameras (clusters) with replacement.
    Returns: (point_estimate, ci_95, boot_samples, group_results)
    """
    cameras = np.unique(locations)
    cam_idx = {cam: np.where(locations == cam)[0] for cam in cameras}

    boot_f1 = []
    boot_groups = defaultdict(list)

    for _ in range(n_boot):
        sampled_cams = rng.choice(cameras, size=len(cameras), replace=True)
        idx = np.concatenate([cam_idx[c] for c in sampled_cams])
        t_b, p_b = targets[idx], preds[idx]
        boot_f1.append(macro_f1(t_b, p_b))
        if groups:
            for gname, gcls in groups.items():
                gm = np.isin(t_b, list(gcls))
                # NaN, not 0.0: a group absent from this resample is a MISSING
                # observation, not a measured F1 of zero. Appending 0.0 shrinks
                # group CIs toward zero and inflates null mass in the p-value.
                boot_groups[gname].append(
                    group_f1(t_b[gm], p_b[gm], gcls) if gm.sum() > 0 else np.nan
                )

    boot_f1 = np.array(boot_f1)
    point = macro_f1(targets, preds)
    ci = np.percentile(boot_f1, [2.5, 97.5])

    group_out = {}
    if groups:
        for gname, vals in boot_groups.items():
            vals = np.array(vals)
            gmask = np.isin(targets, list(groups[gname]))
            gpt = (
                group_f1(targets[gmask], preds[gmask], groups[gname])
                if gmask.sum() > 0
                else np.nan
            )
            group_out[gname] = (gpt, np.nanpercentile(vals, [2.5, 97.5]), vals)

    return point, ci, boot_f1, group_out


print(f"Cluster bootstrap: {N_BOOTSTRAP} resamples of {n_qual_cams} cameras\n")
clust_results = {}

for model_name, res in filtered_results.items():
    point, ci, samples, groups_out = cluster_bootstrap_f1(
        res["targets"], res["preds"], res["locations"], groups=lvis_groups
    )
    # Sanity check: the point estimate must lie inside its own interval.
    # Violation means the point and the resample statistic are computed over
    # different class sets (see the note on `labels=` above).
    assert ci[0] <= point <= ci[1], (
        f"{model_name}: point {point:.4f} outside CI [{ci[0]:.4f}, {ci[1]:.4f}] "
        "— point estimate and bootstrap statistic are not the same estimand"
    )

    clust_results[model_name] = dict(
        point=point, ci=ci, samples=samples, groups=groups_out
    )
    print(f"{model_name}")
    print(
        f"  F1 macro: {point:.4f}  95% CI [{ci[0]:.4f}, {ci[1]:.4f}]  width={ci[1] - ci[0]:.4f}"
    )
    for gname, (gp, gci, _) in groups_out.items():
        # Same estimand check at the group level.
        assert np.isnan(gp) or gci[0] <= gp <= gci[1], (
            f"{model_name}/{gname}: point {gp:.4f} outside CI "
            f"[{gci[0]:.4f}, {gci[1]:.4f}]"
        )
        print(f"  {gname}: {gp:.4f}  95% CI [{gci[0]:.4f}, {gci[1]:.4f}]")
    print()

cameras = np.unique(filtered_results[MODEL_A]["locations"])
cam_idx_a = {
    cam: np.where(filtered_results[MODEL_A]["locations"] == cam)[0] for cam in cameras
}

pair_gap_results = {}

for pair_i, (m_a, m_b) in enumerate(MODEL_PAIRS):
    res_a = filtered_results[m_a]
    res_b = filtered_results[m_b]

    paired_rng = np.random.default_rng(SEED + 100 + pair_i)
    gap_overall, gap_groups = [], defaultdict(list)

    for _ in range(N_BOOTSTRAP):
        sampled_cams = paired_rng.choice(cameras, size=len(cameras), replace=True)
        idx = np.concatenate([cam_idx_a[c] for c in sampled_cams])
        ta, pa = res_a["targets"][idx], res_a["preds"][idx]
        tb, pb = res_b["targets"][idx], res_b["preds"][idx]
        f1a = macro_f1(ta, pa)
        f1b = macro_f1(tb, pb)
        gap_overall.append(f1a - f1b)
        for gname, gcls in lvis_groups.items():
            gm = np.isin(ta, list(gcls))
            # Both models scored on the same images and the same label set,
            # so the gap stays paired.
            gap_groups[gname].append(
                group_f1(ta[gm], pa[gm], gcls) - group_f1(tb[gm], pb[gm], gcls)
                if gm.sum() > 0
                else np.nan
            )

    gap_overall = np.array(gap_overall)
    ci_gap = np.percentile(gap_overall, [2.5, 97.5])
    point_gap = clust_results[m_a]["point"] - clust_results[m_b]["point"]
    p_raw = boot_p(gap_overall)

    pair_gap_results[(m_a, m_b)] = dict(
        gap=gap_overall,
        ci=ci_gap,
        point=point_gap,
        p_raw=p_raw,
        gap_groups={g: np.array(v) for g, v in gap_groups.items()},
    )

# Holm-Bonferroni correction on overall p-values
raw_ps = [pair_gap_results[pair]["p_raw"] for pair in MODEL_PAIRS]
adjusted = holm_bonferroni(raw_ps)

print("=" * 80)
print(f"PAIRED CLUSTER BOOTSTRAP — ALL PAIRS (Holm-Bonferroni, {N_PAIRS} comparisons)")
print("=" * 80)

for idx, (m_a, m_b) in enumerate(MODEL_PAIRS):
    r = pair_gap_results[(m_a, m_b)]
    adj_p, sig = adjusted[idx]
    mark = "✓" if sig else "✗"
    print(f"\n{m_a}  vs  {m_b}")
    print(f"  F1 gap: {r['point']:+.4f}  95% CI [{r['ci'][0]:+.4f}, {r['ci'][1]:+.4f}]")
    print(f"  p-value (raw): {r['p_raw']:.4f}   p-value (Holm): {adj_p:.4f}")
    print(
        f"  {mark} {'Significant' if sig else 'NOT significant'} at α=0.05 (Holm-corrected)"
    )

    # Per-group gaps. NOTE: these are exploratory — two-sided via the CI, but
    # NOT included in the Holm correction, which covers overall pairs only.
    for gname in lvis_groups:
        g_arr = r["gap_groups"][gname]
        n_drop = int(np.isnan(g_arr).sum())
        g_ci = np.nanpercentile(g_arr, [2.5, 97.5])
        gp = (
            clust_results[m_a]["groups"][gname][0]
            - clust_results[m_b]["groups"][gname][0]
        )
        g_p = boot_p(g_arr)
        s = "✓" if (g_ci[0] > 0 or g_ci[1] < 0) else "✗"
        note = f"  [{n_drop}/{N_BOOTSTRAP} resamples dropped]" if n_drop else ""
        print(
            f"  {s} {gname}: gap={gp:+.4f}  CI [{g_ci[0]:+.4f}, {g_ci[1]:+.4f}]  p={g_p:.4f}{note}"
        )

In [ ]:
# PER-CAMERA WILCOXON SIGNED-RANK TEST
# For each qualifying camera: compute F1-macro for both models.
# All pairwise comparisons with Holm-Bonferroni correction.
unique_cams = [
    c
    for c in np.unique(filtered_results[MODEL_A]["locations"])
    if cam_counts[c] >= MIN_CAM_IMAGES
]

print(f"Cameras used (>={MIN_CAM_IMAGES} val images): {len(unique_cams)}")

wilcoxon_results = {}
raw_ps_wilcox = []

for m_a, m_b in MODEL_PAIRS:
    res_a = filtered_results[m_a]
    res_b = filtered_results[m_b]

    cam_f1_a, cam_f1_b, cam_n, cam_gaps = [], [], [], []

    for cam in unique_cams:
        mask = res_a["locations"] == cam
        ta, pa = res_a["targets"][mask], res_a["preds"][mask]
        tb, pb = res_b["targets"][mask], res_b["preds"][mask]

        if len(set(ta)) < 2:
            continue

        f1a = f1_score(ta, pa, average="macro", zero_division=0)
        f1b = f1_score(tb, pb, average="macro", zero_division=0)
        cam_f1_a.append(f1a)
        cam_f1_b.append(f1b)
        cam_n.append(mask.sum())
        cam_gaps.append(f1a - f1b)

    cam_f1_a = np.array(cam_f1_a)
    cam_f1_b = np.array(cam_f1_b)
    cam_gaps = np.array(cam_gaps)
    cam_n = np.array(cam_n)

    wilcox = stats.wilcoxon(cam_f1_a, cam_f1_b, alternative="two-sided")
    raw_ps_wilcox.append(wilcox.pvalue)

    wilcoxon_results[(m_a, m_b)] = dict(
        f1_a=cam_f1_a,
        f1_b=cam_f1_b,
        gaps=cam_gaps,
        cam_n=cam_n,
        stat=wilcox.statistic,
        p_raw=wilcox.pvalue,
        median_a=np.median(cam_f1_a),
        median_b=np.median(cam_f1_b),
        median_gap=np.median(cam_gaps),
        wins=(cam_gaps > 0).sum(),
        total=len(cam_gaps),
    )

# Holm-Bonferroni correction
adjusted_wilcox = holm_bonferroni(raw_ps_wilcox)

print("\n" + "=" * 80)
print(f"PER-CAMERA WILCOXON — ALL PAIRS (Holm-Bonferroni, {N_PAIRS} comparisons)")
print("=" * 80)

for idx, (m_a, m_b) in enumerate(MODEL_PAIRS):
    r = wilcoxon_results[(m_a, m_b)]
    adj_p, sig = adjusted_wilcox[idx]
    mark = "✓" if sig else "✗"
    print(f"\n{m_a}  vs  {m_b}")
    print(f"  Cameras evaluated:    {r['total']}")
    print(f"  Median F1 — {m_a}: {r['median_a']:.4f}")
    print(f"  Median F1 — {m_b}: {r['median_b']:.4f}")
    print(f"  Median gap:           {r['median_gap']:+.4f}")
    print(f"  {m_a} wins:  {r['wins']} / {r['total']} cameras")
    print(f"  Wilcoxon (raw):  stat={r['stat']:.1f}, p={r['p_raw']:.4f}")
    print(f"  Wilcoxon (Holm): p={adj_p:.4f}")
    print(
        f"  {mark} {'Significant' if sig else 'NOT significant'} at α=0.05 (Holm-corrected)"
    )

In [ ]:
# McNEMAR'S TEST  (paired binary, image level)
# All pairwise comparisons with Holm-Bonferroni correction.

mcnemar_results = {}
raw_ps_mcnemar = []

for m_a, m_b in MODEL_PAIRS:
    ca = filtered_results[m_a]["preds"] == filtered_results[m_a]["targets"]
    cb = filtered_results[m_b]["preds"] == filtered_results[m_b]["targets"]
    total = len(ca)

    n00 = (ca & cb).sum()
    n01 = (ca & ~cb).sum()  # A right, B wrong
    n10 = (~ca & cb).sum()  # A wrong, B right
    n11 = (~ca & ~cb).sum()

    table = np.array([[n00, n01], [n10, n11]])
    result = mcnemar(table, exact=False, correction=True)
    raw_ps_mcnemar.append(result.pvalue)

    mcnemar_results[(m_a, m_b)] = dict(
        n00=n00,
        n01=n01,
        n10=n10,
        n11=n11,
        total=total,
        stat=result.statistic,
        p_raw=result.pvalue,
    )

# Holm-Bonferroni correction
adjusted_mcnemar = holm_bonferroni(raw_ps_mcnemar)

print("=" * 80)
print(f"McNEMAR'S TEST — ALL PAIRS (Holm-Bonferroni, {N_PAIRS} comparisons)")
print("=" * 80)

for idx, (m_a, m_b) in enumerate(MODEL_PAIRS):
    r = mcnemar_results[(m_a, m_b)]
    adj_p, sig = adjusted_mcnemar[idx]
    mark = "✓" if sig else "✗"
    print(f"\n{m_a}  vs  {m_b}")
    print(f"  Total images (eligible): {r['total']}")
    print(f"  Both correct:            {r['n00']:6d}  ({r['n00'] / r['total']:.1%})")
    print(f"  {m_a} only correct: {r['n01']:6d}  ({r['n01'] / r['total']:.1%})")
    print(f"  {m_b} only correct: {r['n10']:6d}  ({r['n10'] / r['total']:.1%})")
    print(f"  Both wrong:              {r['n11']:6d}  ({r['n11'] / r['total']:.1%})")
    print(f"  Statistic: {r['stat']:.2f}")
    print(f"  p-value (raw):  {r['p_raw']:.2e}")
    print(f"  p-value (Holm): {adj_p:.2e}")
    if sig:
        winner = m_a if r["n01"] > r["n10"] else m_b
        margin = abs(r["n01"] - r["n10"])
        print(
            f"  {mark} Significant (Holm-corrected): {winner} gets {margin} more images right"
        )
    else:
        print(f"  {mark} NOT significant at α=0.05 (Holm-corrected)")

In [ ]:
print("=" * 70)
print("RARE CLASS DESCRIPTIVE ANALYSIS")
print("=" * 70)

for model_name in [MODEL_A, MODEL_B]:
    res = filtered_results[model_name]
    rare_mask = np.isin(res["targets"], list(rare_classes))
    t_rare = res["targets"][rare_mask]
    p_rare = res["preds"][rare_mask]

    # Per-class breakdown
    print(f"\n{model_name}:")
    print(f"  Total rare samples: {rare_mask.sum()}")
    print(f"  Accuracy: {(t_rare == p_rare).mean():.3f}")
    print(
        f"  Macro-F1: {f1_score(t_rare, p_rare, average='macro', zero_division=0):.4f}"
    )

    # Per-class detail
    print(f"  {'Class':>8} {'Train n':>8} {'Val n':>8} {'Correct':>8} {'Acc':>8}")
    print(f"  {'-' * 44}")
    for cls in sorted(rare_classes & set(t_rare)):
        cls_mask = t_rare == cls
        n_val = cls_mask.sum()
        n_correct = (p_rare[cls_mask] == cls).sum()
        n_train = train_class_dist.get(cls, 0)
        print(
            f"  {cls:>8} {n_train:>8} {n_val:>8} {n_correct:>8} {n_correct / n_val:>8.2f}"
        )

# What does each model predict for rare-class images?
print("\n── Where do rare-class errors go? ──")
for model_name in [MODEL_A, MODEL_B]:
    res = filtered_results[model_name]
    rare_mask = np.isin(res["targets"], list(rare_classes))
    t_rare = res["targets"][rare_mask]
    p_rare = res["preds"][rare_mask]
    wrong = t_rare != p_rare
    if wrong.sum() > 0:
        pred_in_frequent = np.isin(p_rare[wrong], list(frequent_classes)).mean()
        pred_in_common = np.isin(p_rare[wrong], list(common_classes)).mean()
        pred_in_rare = np.isin(p_rare[wrong], list(rare_classes)).mean()
        print(f"{model_name} misclassifies rare images as:")
        print(f"  Frequent class: {pred_in_frequent:.1%}")
        print(f"  Common class:   {pred_in_common:.1%}")
        print(f"  Other rare:     {pred_in_rare:.1%}")

# Extra

In [ ]:
import pandas as pd


def compute_all_image_metrics(preds, targets):
    """Compute all image-level metrics for one model."""
    results = {}

    # 1. Primary: Macro-F1 (all eligible classes)
    results["Macro-F1"] = f1_score(targets, preds, average="macro", zero_division=0)

    # 2. Blank-F1
    label_list = sorted(eligible_classes)
    label_to_idx = {c: i for i, c in enumerate(label_list)}
    per_class_f1 = f1_score(
        targets, preds, labels=label_list, average=None, zero_division=0
    )
    if BLANK_CLASS in label_to_idx:
        results["Blank-F1"] = per_class_f1[label_to_idx[BLANK_CLASS]]
    else:
        results["Blank-F1"] = None

    # 3. Species-F1 (macro, excluding non-species)
    sp_labels = sorted(species_classes)
    sp_mask = np.isin(targets, sp_labels)
    if sp_mask.sum() > 0:
        results["Species-F1"] = f1_score(
            targets[sp_mask],
            preds[sp_mask],
            labels=sp_labels,
            average="macro",
            zero_division=0,
        )
    else:
        results["Species-F1"] = None

    # 4. LVIS segment breakdown
    for seg_name, seg_cls in lvis_segments.items():
        seg_labels = sorted(seg_cls)
        seg_mask = np.isin(targets, seg_labels)
        if seg_mask.sum() > 0:
            results[seg_name] = f1_score(
                targets[seg_mask],
                preds[seg_mask],
                labels=seg_labels,
                average="macro",
                zero_division=0,
            )
        else:
            results[seg_name] = None

    return results


metrics_table = {}
for model_name, r in all_results.items():
    metrics_table[model_name] = compute_all_image_metrics(r["preds"], r["targets"])

df = pd.DataFrame(metrics_table).T

col_order = [
    "Macro-F1",
    "Blank-F1",
    "Species-F1",
    "Head (>100)",
    "Medium (10–100)",
    "Tail (<10)",
]
df = df[col_order]

print("=" * 80)
print("IMAGE-LEVEL DESCRIPTIVE METRICS")
print("=" * 80)
print(f"Eligible classes: {len(eligible_classes)}")
print(f"Species classes:  {len(species_classes)} (excl. {NON_SPECIES})")
print(
    f"Segments — Head: {len(frequent_classes)}, Medium: {len(common_classes)}, Tail: {len(rare_classes)}"
)
print()
print(df.round(4).to_string())

print("\n\nBest per metric:")
for col in df.columns:
    best_model = df[col].idxmax()
    print(f"  {col}: {best_model} ({df[col].max():.4f})")

print("\n\n% ── LaTeX table ──────────────────────────────────────")
print(r"""
\begin{table}[htbp]
\centering
\small
\caption{Métricas descriptivas a nivel de imagen por modelo.
         Macro-F1 es la métrica principal; Blank-F1 y Species-F1
         descomponen el rendimiento por rol funcional
         (Sección~\ref{sec:metricas_rol}). Los segmentos de frecuencia
         siguen los umbrales LVIS (Sección~\ref{sec:segmentos_frecuencia}).
         \textbf{Negrita} indica el mejor valor por columna.}
\label{tab:ciclo1_descriptive}
\begin{tabular}{@{}lccc|ccc@{}}
\toprule
& \multicolumn{3}{c}{\textbf{Rol funcional}} & \multicolumn{3}{c}{\textbf{Segmento de frecuencia}} \\
\cmidrule(lr){2-4} \cmidrule(lr){5-7}
\textbf{Modelo} & Macro-F1 & Blank-F1 & Species-F1 & Head & Medium & Tail \\
\midrule""")

# Generate rows with bold for best
for model_name in df.index:
    row_vals = []
    for col in col_order:
        val = df.loc[model_name, col]
        is_best = (val == df[col].max()) and pd.notna(val)
        if pd.isna(val):
            row_vals.append("--")
        elif is_best:
            row_vals.append(f"\\textbf{{{val:.3f}}}")
        else:
            row_vals.append(f"{val:.3f}")

    # Shorten model names for table
    short = model_name.replace("pretrained", "(ImageNet)").replace("timm ", "")
    print(f"{short} & {' & '.join(row_vals)} \\\\")

print(r"""\bottomrule
\end{tabular}
\end{table}""")

In [ ]:
META = Path(
    "/path/to/iwildcam_224/metadata/metadata/iwildcam2022_train_annotations.json"
)

# RE-RUN INFERENCE WITH SEQ_IDS

with open(META) as f:
    metadata = json.load(f)
image_lookup = {img["id"]: img for img in metadata["images"]}

dm.setup("test")
test_loader = dm.test_dataloader()

# Get the underlying dataset to access image IDs
test_dataset = test_loader.dataset
if hasattr(test_dataset, "dataset"):
    full_dataset = test_dataset.dataset
    test_indices = test_dataset.indices
else:
    full_dataset = test_dataset
    test_indices = list(range(len(test_dataset)))

# Build seq_id array aligned with dataloader order
test_seq_ids = np.array(
    [
        image_lookup[full_dataset.valid_image_ids[idx]].get("seq_id", "unknown")
        for idx in test_indices
    ]
)

print(f"Test images: {len(test_seq_ids)}")
print(f"Unique sequences: {len(set(test_seq_ids))}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
all_results = {}

for model_name, ckpt_path in MODELS.items():
    print(f"\nEvaluating {model_name}...")
    model = BaseClassifier.load_from_checkpoint(ckpt_path)
    model.eval().to(device)

    preds_list, targets_list = [], []
    with torch.no_grad():
        for images, targets in tqdm(test_loader, desc=model_name):
            logits = model(images.to(device))
            preds_list.extend(logits.argmax(1).cpu().tolist())
            targets_list.extend(targets.tolist())

    preds = np.array(preds_list)
    targets = np.array(targets_list)

    # Filter to eligible classes
    mask = np.isin(targets, list(eligible_classes))
    preds_e = preds[mask]
    targets_e = targets[mask]
    locs_e = sample_locations[mask] if "sample_locations" in dir() else None
    seq_ids_e = test_seq_ids[mask]

    all_results[model_name] = dict(
        preds=preds_e,
        targets=targets_e,
        locations=locs_e,
        seq_ids=seq_ids_e,
    )

    f1m = f1_score(targets_e, preds_e, average="macro", zero_division=0)
    print(f"  F1 macro (eligible): {f1m:.4f}")
    print(f"  Images (eligible): {len(preds_e)}")
    print(f"  Sequences (eligible): {len(set(seq_ids_e))}")

    del model
    torch.cuda.empty_cache()


def sequence_majority_vote(preds, targets, seq_ids):
    df = pd.DataFrame({"pred": preds, "target": targets, "seq_id": seq_ids})

    def majority(series):
        return Counter(series).most_common(1)[0][0]

    seq_df = (
        df.groupby("seq_id")
        .agg(
            pred=("pred", majority),
            target=("target", "first"),
            count=("pred", "size"),
        )
        .reset_index()
    )
    return seq_df

In [ ]:
def compute_all_sequence_metrics(preds, targets, seq_ids):
    seq_df = sequence_majority_vote(preds, targets, seq_ids)
    s_preds = seq_df["pred"].values
    s_targets = seq_df["target"].values

    results = {}
    results["Macro-F1"] = f1_score(s_targets, s_preds, average="macro", zero_division=0)

    label_list = sorted(eligible_classes)
    per_class_f1 = f1_score(
        s_targets, s_preds, labels=label_list, average=None, zero_division=0
    )
    label_to_idx = {c: i for i, c in enumerate(label_list)}
    if BLANK_CLASS in label_to_idx:
        results["Blank-F1"] = per_class_f1[label_to_idx[BLANK_CLASS]]

    sp_labels = sorted(species_classes)
    sp_mask = np.isin(s_targets, sp_labels)
    if sp_mask.sum() > 0:
        results["Species-F1"] = f1_score(
            s_targets[sp_mask],
            s_preds[sp_mask],
            labels=sp_labels,
            average="macro",
            zero_division=0,
        )

    for seg_name, seg_cls in lvis_segments.items():
        seg_labels = sorted(seg_cls)
        seg_mask = np.isin(s_targets, seg_labels)
        if seg_mask.sum() > 0:
            results[seg_name] = f1_score(
                s_targets[seg_mask],
                s_preds[seg_mask],
                labels=seg_labels,
                average="macro",
                zero_division=0,
            )

    results["Secuencias"] = len(seq_df)
    return results


seq_metrics = {}
for model_name, r in all_results.items():
    seq_metrics[model_name] = compute_all_sequence_metrics(
        r["preds"], r["targets"], r["seq_ids"]
    )

seq_df = pd.DataFrame(seq_metrics).T
col_order = [
    "Secuencias",
    "Macro-F1",
    "Blank-F1",
    "Species-F1",
    "Head (>100)",
    "Medium (10–100)",
    "Tail (<10)",
]
seq_df = seq_df[[c for c in col_order if c in seq_df.columns]]

print("=" * 80)
print("SEQUENCE-LEVEL DESCRIPTIVE METRICS (majority vote)")
print("=" * 80)
print(seq_df.round(4).to_string())

print("\nBest per metric:")
for col in seq_df.columns:
    if col == "Secuencias":
        continue
    best = seq_df[col].idxmax()
    print(f"  {col}: {best} ({seq_df[col].max():.4f})")

In [ ]:
# TOP CONFUSIONS PER MODEL

with open(META) as f:
    metadata = json.load(f)

# original_id -> name
orig_id_to_name = {cat["id"]: cat["name"] for cat in full_ds.categories}

# internal_idx -> original_id
idx_to_orig = {v: k for k, v in full_ds.category_to_idx.items()}

# internal_idx -> name (this is what the model uses)
cat_lookup = {
    idx: orig_id_to_name.get(orig_id, f"cls_{orig_id}")
    for orig_id, idx in full_ds.category_to_idx.items()
}

print(list(cat_lookup.items())[:10])


def top_confusions(preds, targets, train_dist, cat_lookup, top_n=15):
    """Get top N most frequent misclassifications."""
    errors = [(t, p) for t, p in zip(targets, preds) if t != p]
    counts = Counter(errors)

    rows = []
    for (true_cls, pred_cls), count in counts.most_common(top_n):
        rows.append(
            {
                "True class": cat_lookup.get(true_cls, f"cls_{true_cls}"),
                "Predicted as": cat_lookup.get(pred_cls, f"cls_{pred_cls}"),
                "Errors": count,
                "Train (true)": train_dist.get(true_cls, 0),
                "Train (pred)": train_dist.get(pred_cls, 0),
                "True ID": true_cls,
                "Pred ID": pred_cls,
            }
        )
    return pd.DataFrame(rows)


TOP_N = 15

for model_name, r in all_results.items():
    print("=" * 80)
    print(f"TOP {TOP_N} CONFUSIONS: {model_name}")
    print("=" * 80)

    df = top_confusions(r["preds"], r["targets"], train_class_dist, cat_lookup, TOP_N)
    print(df.to_string(index=False))

    # Summary stats
    total_errors = np.sum(r["preds"] != r["targets"])
    top_errors = df["Errors"].sum()
    print(f"\nTotal errors: {total_errors}")
    print(
        f"Top {TOP_N} confusions account for: {top_errors} ({top_errors / total_errors:.1%})"
    )
    print()

print("=" * 80)
print("CONFUSIONS SHARED ACROSS ALL MODELS (top 10)")
print("=" * 80)

confusion_counts = {}
for model_name, r in all_results.items():
    errors = [(t, p) for t, p in zip(r["targets"], r["preds"]) if t != p]
    confusion_counts[model_name] = Counter(errors)

# Find pairs that appear in top 30 of every model
top_per_model = [set(dict(c.most_common(30)).keys()) for c in confusion_counts.values()]
shared = set.intersection(*top_per_model)

rows = []
for true_cls, pred_cls in shared:
    row = {
        "True class": cat_lookup.get(true_cls, f"cls_{true_cls}"),
        "Predicted as": cat_lookup.get(pred_cls, f"cls_{pred_cls}"),
        "Train (true)": train_class_dist.get(true_cls, 0),
    }
    for model_name, counts in confusion_counts.items():
        short = model_name.replace("pretrained", "").replace("timm ", "").strip()
        row[short] = counts.get((true_cls, pred_cls), 0)
    rows.append(row)

shared_df = pd.DataFrame(rows)
if len(shared_df) > 0:
    shared_df = shared_df.sort_values(
        list(shared_df.columns[3:]), ascending=False
    ).head(10)
    print(shared_df.to_string(index=False))
else:
    print("No shared confusions in top 30 of all models.")

In [ ]:
# SHARED CONFUSIONS TABLE WITH PERCENTAGES

# Use targets from any model (they're all the same)
ref_model = list(all_results.keys())[0]
test_class_counts = Counter(all_results[ref_model]["targets"].tolist())

confusion_counts = {}
for model_name, r in all_results.items():
    errors = [(t, p) for t, p in zip(r["targets"], r["preds"]) if t != p]
    confusion_counts[model_name] = Counter(errors)

top_per_model = [set(dict(c.most_common(30)).keys()) for c in confusion_counts.values()]
shared = set.intersection(*top_per_model)

model_names = list(MODELS.keys())
model_short = {
    "DINOv2 ViT-B/14": "DINOv2",
    "ResNet50 pretrained": "ResNet50",
    "timm ViT-B/16": "ViT-B/16",
}

rows = []
for true_cls, pred_cls in shared:
    n_test = test_class_counts.get(true_cls, 0)
    n_train = train_class_dist.get(true_cls, 0)
    true_name = cat_lookup.get(true_cls, f"cls_{true_cls}")
    pred_name = cat_lookup.get(pred_cls, f"cls_{pred_cls}")

    row = {
        "true_cls": true_cls,
        "pred_cls": pred_cls,
        "True class": true_name,
        "Predicted as": pred_name,
        "Train": n_train,
        "Test": n_test,
    }
    total = 0
    for model_name in model_names:
        count = confusion_counts[model_name].get((true_cls, pred_cls), 0)
        pct = (count / n_test * 100) if n_test > 0 else 0
        short = model_short[model_name]
        row[f"{short}_n"] = count
        row[f"{short}_%"] = pct
        total += count
    row["total"] = total
    rows.append(row)

df = pd.DataFrame(rows).sort_values("total", ascending=False).head(10)
# Average confusion % across models as sort key
df["avg_pct"] = (df["DINOv2_%"] + df["ResNet50_%"] + df["ViT-B/16_%"]) / 3
df = df.sort_values("avg_pct", ascending=False).head(10)

print("=" * 100)
print("SHARED CONFUSIONS — TOP 10")
print("=" * 100)
for _, row in df.iterrows():
    print(
        f"  {row['True class']:30s} → {row['Predicted as']:25s}  "
        f"Train={row['Train']:>6,}  Test={row['Test']:>5,}  "
        f"DINOv2={row['DINOv2_n']:>3} ({row['DINOv2_%']:>4.1f}%)  "
        f"ResNet50={row['ResNet50_n']:>3} ({row['ResNet50_%']:>4.1f}%)  "
        f"ViT-B/16={row['ViT-B/16_n']:>3} ({row['ViT-B/16_%']:>4.1f}%)"
    )


def shorten_species(name):
    """Shorten species name for table: 'meleagris ocellata' -> 'M. ocellata'"""
    parts = name.strip().split()
    if len(parts) == 2:
        return f"{parts[0][0].upper()}.~{parts[1]}"
    elif len(parts) == 1:
        return name
    else:
        return f"{parts[0][0].upper()}.~{' '.join(parts[1:])}"


def format_species_latex(name):
    """Format species name in italics for LaTeX."""
    if name == "empty":
        return "empty"
    short = shorten_species(name)
    # Handle "sp" or "sp." at the end
    if short.endswith(" sp") or short.endswith(" sp."):
        base = short.rsplit(" ", 1)[0]
        return f"\\textit{{{base}}} sp."
    return f"\\textit{{{short}}}"


print("\n\n% ── LaTeX table ──────────────────────────────────────")
print(r"""\begin{table}[htbp]
  \centering
  \small
  \caption{Confusiones compartidas entre los tres modelos (top~10).
           Se muestran los pares verdadero--predicho donde los tres
           modelos cometen errores. Las columnas de cada modelo indican
           el porcentaje de imágenes de test de la clase verdadera
           confundidas con la clase predicha ($n$ / \% del total de
           test de esa clase).}
  \label{tab:ciclo1_shared_confusions}
  \begin{tabular}{@{}ll rr rrr@{}}
    \toprule
    & & & & \multicolumn{3}{c}{\textbf{\% confundido}} \\
    \cmidrule(lr){5-7}
    \textbf{Clase verdadera} & \textbf{Predicha como}
      & \textbf{Train} & \textbf{Test}
      & \textbf{DINOv2} & \textbf{ResNet50} & \textbf{ViT-B/16} \\
    \midrule""")

for _, row in df.iterrows():
    true_latex = format_species_latex(row["True class"])
    pred_latex = format_species_latex(row["Predicted as"])
    train_str = f"{row['Train']:,}".replace(",", "{,}")
    test_str = f"{row['Test']:,}".replace(",", "{,}")

    print(
        f"    {true_latex:30s} & {pred_latex:25s} "
        f"& {train_str} & {test_str} "
        f"& {row['DINOv2_%']:.1f} "
        f"& {row['ResNet50_%']:.1f} "
        f"& {row['ViT-B/16_%']:.1f} \\\\"
    )

print(r"""    \bottomrule
  \end{tabular}
\end{table}""")

In [ ]:
# ERRORES DE ALTA CONFIANZA + EJEMPLOS DE CONFUSIÓN

from pathlib import Path

import torch
import torch.nn.functional as F
from PIL import Image

plt.rcParams.update(
    {
        "font.family": "serif",
        "font.size": 9,
    }
)

IMAGE_DIR = Path("/path/to/iwildcam_224/train/train")

MODEL_TO_ANALYZE = "DINOv2 ViT-B/14"

model = BaseClassifier.load_from_checkpoint(MODELS[MODEL_TO_ANALYZE])
model.eval().to(device)

all_probs = []
all_preds = []
all_targets = []
all_img_ids = []

with torch.no_grad():
    batch_idx = 0
    for images, targets in tqdm(test_loader, desc="Recolectando confianzas"):
        logits = model(images.to(device))
        probs = F.softmax(logits, dim=1)
        max_probs, preds = probs.max(dim=1)

        all_probs.extend(max_probs.cpu().tolist())
        all_preds.extend(preds.cpu().tolist())
        all_targets.extend(targets.tolist())

        start = batch_idx * test_loader.batch_size
        end = start + len(targets)
        batch_img_ids = [
            full_ds.valid_image_ids[test_indices[i]] for i in range(start, end)
        ]
        all_img_ids.extend(batch_img_ids)
        batch_idx += 1

del model
torch.cuda.empty_cache()

all_probs = np.array(all_probs)
all_preds = np.array(all_preds)
all_targets = np.array(all_targets)
all_img_ids = np.array(all_img_ids)

# Filtrar a clases elegibles
mask = np.isin(all_targets, list(eligible_classes))
probs_e = all_probs[mask]
preds_e = all_preds[mask]
targets_e = all_targets[mask]
img_ids_e = all_img_ids[mask]

errors = preds_e != targets_e
error_probs = probs_e[errors]
error_preds = preds_e[errors]
error_targets = targets_e[errors]
error_img_ids = img_ids_e[errors]

top_conf_idx = np.argsort(error_probs)[::-1][:20]

print("=" * 80)
print(f"TOP 20 ERRORES DE ALTA CONFIANZA: {MODEL_TO_ANALYZE}")
print("=" * 80)
for i in top_conf_idx:
    true_name = cat_lookup.get(error_targets[i], f"cls_{error_targets[i]}")
    pred_name = cat_lookup.get(error_preds[i], f"cls_{error_preds[i]}")
    print(
        f"  conf={error_probs[i]:.3f}  real={true_name:30s}  pred={pred_name:30s}  img={error_img_ids[i]}"
    )


def show_error_grid(
    img_ids,
    true_labels,
    pred_labels,
    confidences,
    cat_lookup,
    image_dir,
    title,
    n_cols=5,
    n_rows=1,
):
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3 * n_cols, 4 * n_rows))
    if n_rows == 1:
        axes = [axes] if n_cols == 1 else list(axes)
    else:
        axes = axes.flatten()

    for i, ax in enumerate(axes):
        if i >= len(img_ids):
            ax.axis("off")
            continue

        img_path = image_dir / f"{img_ids[i]}.jpg"
        if img_path.exists():
            img = Image.open(img_path)
            ax.imshow(img)
        else:
            ax.text(0.5, 0.5, "No encontrada", ha="center", va="center")

        true_name = cat_lookup.get(true_labels[i], f"cls_{true_labels[i]}")
        pred_name = cat_lookup.get(pred_labels[i], f"cls_{pred_labels[i]}")

        ax.set_title(
            f"Real: {true_name}\nPred: {pred_name}\nConfianza: {confidences[i]:.2f}",
            fontsize=7,
            fontstyle="italic",
            pad=8,
        )
        ax.axis("off")

    fig.suptitle(title, fontsize=11, fontweight="bold", y=1.02)
    plt.tight_layout()
    return fig


# Usá los internal IDs de tu dataset, verificá que coincidan
TOP_CONFUSION_PAIRS = [
    (125, 0),  # macaca sp -> empty
    (163, 164),  # meleagris ocellata -> crax rubra
    (164, 163),  # crax rubra -> meleagris ocellata (bidireccional)
    (15, 8),  # panthera onca -> leopardus pardalis
]

for true_cls, pred_cls in TOP_CONFUSION_PAIRS:
    pair_mask = (error_targets == true_cls) & (error_preds == pred_cls)
    pair_img_ids = error_img_ids[pair_mask]
    pair_probs = error_probs[pair_mask]
    pair_targets = error_targets[pair_mask]
    pair_preds = error_preds[pair_mask]

    if len(pair_img_ids) == 0:
        print(f"No errors found for {true_cls} -> {pred_cls}")
        continue

    sort_idx = np.argsort(pair_probs)[::-1][:10]

    true_name = cat_lookup.get(true_cls, f"cls_{true_cls}")
    pred_name = cat_lookup.get(pred_cls, f"cls_{pred_cls}")
    n_test = test_class_counts.get(true_cls, 0)
    pct = pair_mask.sum() / n_test * 100 if n_test > 0 else 0

    fig = show_error_grid(
        pair_img_ids[sort_idx],
        pair_targets[sort_idx],
        pair_preds[sort_idx],
        pair_probs[sort_idx],
        cat_lookup,
        IMAGE_DIR,
        f"{true_name} \u2192 {pred_name} ({pair_mask.sum()} errores, {pct:.1f}% de test)",
    )
    fname = f"confusion_{true_cls}_{pred_cls}.pdf"
    plt.savefig(OUTPUT_DIR / fname, bbox_inches="tight")
    plt.show()

In [ ]:
for model_name, r in all_results.items():
    error_mask = r["preds"] != r["targets"]
    error_preds = r["preds"][error_mask]
    error_targets = r["targets"][error_mask]

    n_errors = error_mask.sum()
    n_to_empty = (error_preds == BLANK_CLASS).sum()
    n_to_species = n_errors - n_to_empty

    # Also: errors FROM empty to species
    n_from_empty = (error_targets == BLANK_CLASS).sum()

    print(f"{model_name}:")
    print(f"  Total errors: {n_errors}")
    print(f"  Pred empty (false negative): {n_to_empty} ({n_to_empty / n_errors:.1%})")
    print(
        f"  Pred species (species confusion): {n_to_species} ({n_to_species / n_errors:.1%})"
    )
    print(
        f"  True empty, pred species (false positive): {n_from_empty} ({n_from_empty / n_errors:.1%})"
    )
    print()

In [ ]:
for model_name, r in all_results.items():
    error_mask = r["preds"] != r["targets"]
    error_preds = r["preds"][error_mask]
    error_targets = r["targets"][error_mask]

    n_errors = error_mask.sum()

    # Where do wrong predictions land?
    pred_to_head = sum(1 for p in error_preds if p in frequent_classes)
    pred_to_mid = sum(1 for p in error_preds if p in common_classes)
    pred_to_tail = sum(1 for p in error_preds if p in rare_classes)
    pred_to_empty = sum(1 for p in error_preds if p == BLANK_CLASS)

    # Where do errors come from?
    true_from_head = sum(1 for t in error_targets if t in frequent_classes)
    true_from_mid = sum(1 for t in error_targets if t in common_classes)
    true_from_tail = sum(1 for t in error_targets if t in rare_classes)
    true_from_empty = sum(1 for t in error_targets if t == BLANK_CLASS)

    print(f"{model_name} ({n_errors} errors):")
    print("  Errors PREDICTED AS:")
    print(f"    Empty:    {pred_to_empty:>5} ({pred_to_empty / n_errors:>5.1%})")
    print(f"    Head:     {pred_to_head:>5} ({pred_to_head / n_errors:>5.1%})")
    print(f"    Medium:   {pred_to_mid:>5} ({pred_to_mid / n_errors:>5.1%})")
    print(f"    Tail:     {pred_to_tail:>5} ({pred_to_tail / n_errors:>5.1%})")
    print("  Errors COMING FROM:")
    print(f"    Empty:    {true_from_empty:>5} ({true_from_empty / n_errors:>5.1%})")
    print(f"    Head:     {true_from_head:>5} ({true_from_head / n_errors:>5.1%})")
    print(f"    Medium:   {true_from_mid:>5} ({true_from_mid / n_errors:>5.1%})")
    print(f"    Tail:     {true_from_tail:>5} ({true_from_tail / n_errors:>5.1%})")
    print()

In [ ]:
for model_name, r in all_results.items():
    error_mask = r["preds"] != r["targets"]
    error_preds = r["preds"][error_mask]
    error_targets = r["targets"][error_mask]

    n_errors = error_mask.sum()

    # Separate empty from frequency segments
    species_head = frequent_classes - {BLANK_CLASS}

    # Where do wrong predictions land?
    pred_to_empty = sum(1 for p in error_preds if p == BLANK_CLASS)
    pred_to_head = sum(1 for p in error_preds if p in species_head)
    pred_to_mid = sum(1 for p in error_preds if p in common_classes)
    pred_to_tail = sum(1 for p in error_preds if p in rare_classes)

    # Where do errors come from?
    true_from_empty = sum(1 for t in error_targets if t == BLANK_CLASS)
    true_from_head = sum(1 for t in error_targets if t in species_head)
    true_from_mid = sum(1 for t in error_targets if t in common_classes)
    true_from_tail = sum(1 for t in error_targets if t in rare_classes)

    print(f"{model_name} ({n_errors} errors):")
    print("  Errors PREDICTED AS:")
    print(f"    Empty:    {pred_to_empty:>5} ({pred_to_empty / n_errors:>5.1%})")
    print(f"    Head:     {pred_to_head:>5} ({pred_to_head / n_errors:>5.1%})")
    print(f"    Medium:   {pred_to_mid:>5} ({pred_to_mid / n_errors:>5.1%})")
    print(f"    Tail:     {pred_to_tail:>5} ({pred_to_tail / n_errors:>5.1%})")
    print("  Errors COMING FROM:")
    print(f"    Empty:    {true_from_empty:>5} ({true_from_empty / n_errors:>5.1%})")
    print(f"    Head:     {true_from_head:>5} ({true_from_head / n_errors:>5.1%})")
    print(f"    Medium:   {true_from_mid:>5} ({true_from_mid / n_errors:>5.1%})")
    print(f"    Tail:     {true_from_tail:>5} ({true_from_tail / n_errors:>5.1%})")

    # Sanity check
    total_pred = pred_to_empty + pred_to_head + pred_to_mid + pred_to_tail
    print(f"  Sum check: {total_pred} / {n_errors}")
    print()

In [ ]:
ref = all_results[list(MODELS.keys())[0]]["targets"]
n_total = len(ref)

n_empty = sum(1 for t in ref if t == BLANK_CLASS)
n_head = sum(1 for t in ref if t in species_head)
n_mid = sum(1 for t in ref if t in common_classes)
n_tail = sum(1 for t in ref if t in rare_classes)

print("Test set composition:")
print(f"  Empty:  {n_empty:>6} ({n_empty / n_total:.1%})")
print(f"  Head:   {n_head:>6} ({n_head / n_total:.1%})")
print(f"  Medium: {n_mid:>6} ({n_mid / n_total:.1%})")
print(f"  Tail:   {n_tail:>6} ({n_tail / n_total:.1%})")

print("\nBias ratio (pred% / base%):")
for model_name, r in all_results.items():
    error_mask = r["preds"] != r["targets"]
    error_preds = r["preds"][error_mask]
    n_errors = error_mask.sum()

    pred_empty = sum(1 for p in error_preds if p == BLANK_CLASS) / n_errors
    pred_head = sum(1 for p in error_preds if p in species_head) / n_errors
    pred_mid = sum(1 for p in error_preds if p in common_classes) / n_errors
    pred_tail = sum(1 for p in error_preds if p in rare_classes) / n_errors

    print(f"  {model_name}:")
    print(
        f"    Empty:  pred={pred_empty:.1%}  base={n_empty / n_total:.1%}  ratio={pred_empty / (n_empty / n_total):.2f}"
    )
    print(
        f"    Head:   pred={pred_head:.1%}  base={n_head / n_total:.1%}  ratio={pred_head / (n_head / n_total):.2f}"
    )
    print(
        f"    Medium: pred={pred_mid:.1%}  base={n_mid / n_total:.1%}  ratio={pred_mid / (n_mid / n_total):.2f}"
    )
    print(
        f"    Tail:   pred={pred_tail:.1%}  base={n_tail / n_total:.1%}  ratio={pred_tail / (n_tail / n_total):.2f}"
    )